# Welcome to the Backend!

This notebook is executed in 4 steps:
1. Installing dependencies
2. Setting up the RAG-based model
3. Setting up the servers, including the: </br>
    3a. Flask server (to connect to frontened) </br>
    3b. ngrok tunnel (to connect to Collab by forwarding traffic to external server) </br>
4. Passing the ngrok URL to the frontend!

## 1. Install Dependencies

In [16]:
!pip install flask flask-cors transformers accelerate pyngrok
!pip install -U bitsandbytes>=0.46.1

For the model.

In [17]:
import gc
import torch
import sys
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

For Flask.

In [18]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading

## 2. Set up the RAG-Based Model

Clean the environment.

In [19]:
for name in ["model", "processor", "tokenizer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

!rm -rf /root/.cache/huggingface/hub
!rm -rf /root/.cache/torch
!rm -rf /root/.cache/transformers
!rm -rf /root/.cache/pip

!df -h
!nvidia-smi # ensures NVIDIA drivers are installed and visible

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   44G  193G  19% /
tmpfs            64M     0   64M   0% /dev
shm              41G  4.0K   41G   1% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       242G   61G  182G  26% /opt/bin/.nvidia
tmpfs            42G   92K   42G   1% /var/colab
tmpfs            42G     0   42G   0% /proc/acpi
tmpfs            42G     0   42G   0% /proc/scsi
tmpfs            42G     0   42G   0% /sys/firmware
Thu May  7 04:03:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|              

Check GPU visibility.

In [20]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU", torch.cuda.get_device_name(0))
  print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
  print("GPU memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")
else:
  print("No GPU found. Go to Runtime > Change runtime type > GPU")

CUDA available: True
GPU NVIDIA A100-SXM4-40GB
GPU memory allocated: 0.0 GB
GPU memory reserved: 0.0 GB


Load in model.

In [21]:
SYSTEM_PROMPT = """
You are Lisan al-GPT, a Dune-focused question-answering chatbot.

For this baseline experiment, answer using only your pretrained knowledge.
Do not claim to have access to the books, retrieved passages, a database, or citations.
If you are uncertain, say so clearly.
If the user asks for no spoilers past a certain book or chapter, respect that spoiler boundary.
Keep answers concise but informative.
"""

In [22]:
print("Loading model on GPU..." if torch.cuda.is_available() else "Loading on CPU...")

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="auto"
)

Loading model on GPU...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [23]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, device_map='auto')
model.eval()
# ensure model was loaded correctly
print("Loaded:", MODEL_ID)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


In [24]:
def generate(prompt: str, max_new_tokens=300, temperature=0.2) -> str:
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    device = next(model.parameters()).device
    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
    }

    if temperature > 0:
        generation_kwargs["temperature"] = temperature

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            **generation_kwargs
        )
    generated_ids_trimmed = generated_ids[:, inputs.input_ids.shape[1]:]
    output_text = tokenizer.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0]

    return output_text

Check to make sure the model fits on the GPU.

In [25]:
# check to make sure everything fits on GPU
devices = set(param.device for param in model.parameters());

# Compare against an actual device object
if devices == {torch.device('cuda', 0)}:
    print("PASSED: All parameters are on GPU.")
elif all(d.type == 'cuda' for d in devices):
    print("PASSED: All parameters are on GPU.")
else:
    print("FAILED: Some parameters are NOT on GPU.")
    # Show which devices are being used
    print("Devices found:", devices)
    sys.exit() # quit if not

PASSED: All parameters are on GPU.


## 3a. Flask Server

In [26]:
app = Flask(__name__)
CORS(app)

@app.route("/generate", methods=["POST"])
def generate_route():
    data = request.json
    prompt = data.get("prompt", "")
    response = generate(prompt)
    return jsonify({"response": response})

# Start Flask in a background thread (so the notebook doesn't freeze)
threading.Thread(target=lambda: app.run(port=5000)).start()

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


## 3b. ngrok Tunnel

In [ ]:
auth_token="DONT PUSH THIS TO THE REPO"
ngrok.set_auth_token(auth_token)

# Open a public tunnel to port 5000
public_url = ngrok.connect(5000)
print("Your public API URL=", public_url)

Your public API URL= NgrokTunnel: "https://decency-reference-mustang.ngrok-free.dev" -> "http://localhost:5000"


INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:06:15] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:06:22] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:07:41] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:07:45] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:07:56] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:08:04] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:14:53] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:14:56] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:15:09] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:15:11] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:15:22] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 04:15:26] "POST /generate HT